# Error analysis: foundation models vs CNN
**Owner: Srija** · Neural Band instrument recognition

Finds **which instruments fail**, **what they are confused with**, and **in which conditions** (genre, drums), using the prediction files saved by the earlier runs. No training, so the **CPU runtime is enough**.

**Run the two cells in order:** 1. Setup → 2. Error analysis.

## 1. Setup
Connects Google Drive, gets the repo, unzips the IRMAS audio from Drive and restores your saved results.
Click **Connect** when Colab asks for Drive access. The GitHub token box appears only if the repo is private (press Enter if it's public).
It should end with `clips : 6705`.

In [1]:
import os
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/neural_band'
REPO  = '/content/repo'

if not os.path.exists(REPO):
    from getpass import getpass
    token = getpass('GitHub token (press Enter if the repo is public): ').strip()
    url = (f'https://{token}@' if token else 'https://') + 'github.com/Srija-d/Neural-Band-instrument-recognition-.git'
    os.system(f'git clone -q {url} {REPO}')
os.chdir(REPO)
!pip install -q -r requirements.txt

# IRMAS audio (needed only for the listening step): unzip from Drive if missing
count = lambda: sum(f.endswith('.wav') for _, _, fs in os.walk('data/IRMAS-TrainingData') for f in fs)
if count() != 6705:
    !rm -rf data/IRMAS-TrainingData
    !mkdir -p data
    !cp "{DRIVE}/IRMAS-TrainingData.zip" data/
    !python scripts/download_irmas.py

# Restore your saved results and caches from Drive
!mkdir -p cache/foundation results/foundation
!rsync -a "{DRIVE}/cache_foundation/" cache/foundation/
!rsync -a "{DRIVE}/results_foundation/" results/foundation/

def save_to_drive():
    os.system(f'rsync -a results/foundation/ "{DRIVE}/results_foundation/"')
    os.system(f'rsync -a cache/foundation/ "{DRIVE}/cache_foundation/"')
    print('saved to', DRIVE)

print('folder:', os.getcwd())
print('clips :', count(), '(should be 6705)')
print('prediction files:', len(os.listdir('results/foundation/finetune/preds')) if os.path.exists('results/foundation/finetune/preds') else 0)

Mounted at /content/drive
GitHub token (press Enter if the repo is public): ··········
Extracting...
Done: 6705 wav files in /content/repo/data/IRMAS-TrainingData
folder: /content/repo
clips : 6705 (should be 6705)
prediction files: 24


## 2. Error analysis
Outputs, in order:
1. **F1 per instrument** for CLAP, AST probe and CNN: which instruments are hard
2. **Top confusions**: what each instrument is mistaken for
3. **Accuracy by genre and drums**: in which conditions models fail (from the tags in IRMAS file names)
4. **Confusion matrices** side by side (saved for the report)
5. **5% vs 100% data per instrument** (AST probe): which instruments need more data
6. **Clips to listen to** from the most common confusion: write down what you hear

Results are saved to `results/foundation/error_analysis/` and copied to Drive.

In [2]:
# ===== Error analysis (CPU only): which instruments fail, what they are confused with, and in which conditions =====
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

os.chdir('/content/repo')

CLASSES = ['cel', 'cla', 'flu', 'gac', 'gel', 'org', 'pia', 'sax', 'tru', 'vio', 'voi']
NAMES = {'cel': 'cello', 'cla': 'clarinet', 'flu': 'flute', 'gac': 'acoustic guitar', 'gel': 'electric guitar',
         'org': 'organ', 'pia': 'piano', 'sax': 'saxophone', 'tru': 'trumpet', 'vio': 'violin', 'voi': 'voice'}
GENRES = {'pop_roc': 'pop/rock', 'jaz_blu': 'jazz/blues', 'cla': 'classical', 'cou_fol': 'country/folk', 'lat_sou': 'latin/soul'}
FT, CNN = 'results/foundation/finetune/preds', 'results/cnn/preds'
OUT = 'results/foundation/error_analysis'
os.makedirs(OUT, exist_ok=True)

def load(path):
    p = pd.read_csv(path)
    tags = p.path.str.findall(r'\[([^\]]+)\]')                      # e.g. [cel][nod][cla]
    p['drums'] = tags.map(lambda t: 'drums' if 'dru' in t[1:] else ('no drums' if 'nod' in t[1:] else 'unknown'))
    p['genre'] = tags.map(lambda t: next((GENRES[x] for x in t[1:] if x in GENRES), 'unknown'))
    p['correct'] = p.label == p.pred
    return p

# The models to compare at 100% data (edit if a file is missing)
models = {
    'CLAP fine-tuned': f'{FT}/clap_ft_top2_p100_s0_val.csv',
    'AST probe':       f'{FT}/ast_probe_p100_s0_val.csv',
    'CNN':             f'{CNN}/specaug1_clip3s_p100_s0_val.csv',
}
preds = {name: load(path) for name, path in models.items() if os.path.exists(path)}
print('loaded:', list(preds))

# 1) Per-instrument F1: which instruments are hard, and are they hard for every model?
def f1_per_class(p):
    out = {}
    for c in CLASSES:
        tp = ((p.label == c) & (p.pred == c)).sum(); fp = ((p.label != c) & (p.pred == c)).sum(); fn = ((p.label == c) & (p.pred != c)).sum()
        out[NAMES[c]] = 2 * tp / max(2 * tp + fp + fn, 1)
    return pd.Series(out)
per_class = pd.DataFrame({m: f1_per_class(p) for m, p in preds.items()}).round(3)
per_class['n clips'] = pd.Series({NAMES[c]: (next(iter(preds.values())).label == c).sum() for c in CLASSES})
per_class = per_class.sort_values(list(preds)[0])
print('\n1) F1 per instrument (hardest first, sorted by the first model):'); display(per_class)

# 2) Top confusions: true instrument -> predicted instrument
for m, p in preds.items():
    w = p[~p.correct]
    top = w.groupby(['label', 'pred']).size().sort_values(ascending=False).head(8)
    top.index = [f'{NAMES[a]} -> {NAMES[b]}' for a, b in top.index]
    print(f'\n2) {m}: {len(w)} of {len(p)} val clips wrong. Most common confusions:'); display(top.rename('clips').to_frame())

# 3) Conditions: accuracy by genre and by drums
for col in ['genre', 'drums']:
    t = pd.DataFrame({m: p.groupby(col).correct.mean() for m, p in preds.items()}).round(3)
    t['n clips'] = next(iter(preds.values())).groupby(col).size()
    print(f'\n3) Accuracy by {col}:'); display(t)

# 4) Row-normalised confusion matrices (each row = 100% of that instrument's clips)
fig, axes = plt.subplots(1, len(preds), figsize=(5.2 * len(preds), 4.8))
for ax, (m, p) in zip(np.atleast_1d(axes), preds.items()):
    cm = pd.crosstab(p.label, p.pred).reindex(index=CLASSES, columns=CLASSES, fill_value=0)
    cm = cm.div(cm.sum(axis=1).replace(0, 1), axis=0)
    ax.imshow(cm.values, cmap='Blues', vmin=0, vmax=1)
    ax.set_xticks(range(11)); ax.set_xticklabels(CLASSES, rotation=90); ax.set_yticks(range(11)); ax.set_yticklabels(CLASSES)
    for i in range(11):
        for j in range(11):
            v = cm.values[i, j]
            if v >= 0.05: ax.text(j, i, f'{v:.0%}'[:-1], ha='center', va='center', fontsize=7, color='white' if v > 0.5 else 'black')
    ax.set_title(f'{m} (val, 100% data)'); ax.set_xlabel('predicted'); ax.set_ylabel('true')
fig.tight_layout(); fig.savefig(f'{OUT}/confusion_matrices.png', dpi=150); plt.show()

# 5) Does little data fail on the same instruments? (AST probe: 5% vs 100%)
small, full = f'{FT}/ast_probe_p005_s0_val.csv', f'{FT}/ast_probe_p100_s0_val.csv'
if os.path.exists(small) and os.path.exists(full):
    d = pd.DataFrame({'5% data': f1_per_class(load(small)), '100% data': f1_per_class(load(full))}).round(3)
    d['gain'] = (d['100% data'] - d['5% data']).round(3)
    print('\n5) AST probe F1 per instrument, 5% vs 100% data:'); display(d.sort_values('gain', ascending=False))

# 6) Clips to listen to: the most common confusion of the first model
first = next(iter(preds.values()))
w = first[~first.correct]
a, b = w.groupby(['label', 'pred']).size().idxmax()
listen = w[(w.label == a) & (w.pred == b)].head(5)
print(f'\n6) Listen to these ({NAMES[a]} predicted as {NAMES[b]}). Note what you hear: a second loud instrument? solo vs band? silence?')
display(listen[['path', 'genre', 'drums']])
try:
    from IPython.display import Audio
    for f in listen.path:
        print(f); display(Audio(f'data/IRMAS-TrainingData/{f}'))
except Exception as e:
    print('(audio players skipped:', e, ')')

per_class.to_csv(f'{OUT}/f1_per_instrument.csv')
print(f'\nSaved: {OUT}/confusion_matrices.png and f1_per_instrument.csv')

try:
    save_to_drive()
except NameError:
    print('(run the setup cell first so save_to_drive is defined)')


Output hidden; open in https://colab.research.google.com to view.

## Writing it up (honestly)
- **Report what the numbers show:** the hardest instruments and the most common confusions.
- **Compare models:** e.g. how many trumpet → saxophone errors the CNN makes vs CLAP.
- **Explain with evidence:** describe what you actually hear in the step-6 clips; don't guess beyond it.
- **State limits:** one seed for these runs, validation set only, small genre groups (latin/soul has 3 clips), and most clips have no drums tag.